In [ ]:
import tqdm
import torch
import wandb
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModel
from peft import LoraConfig, get_peft_model
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
wandb.init(project="LoRA MobileBERT - Intrusion Detection Model FineTuninig")

In [ ]:
data = pd.read_csv("../../dataStuff/UNSW_binData.csv")
data

In [ ]:
label_encoder = LabelEncoder()
data["label"] = label_encoder.fit_transform(data["label"])
data

In [ ]:
X = data.drop(columns=["label"]).values
y = data["label"].values.astype(float) 

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = (
    torch.tensor(X_train, dtype=torch.float32).to(device),
    torch.tensor(X_val, dtype=torch.float32).to(device),
    torch.tensor(y_train, dtype=torch.float32).to(device),
    torch.tensor(y_val, dtype=torch.float32).to(device),
)

In [ ]:
batch_size = 16
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

In [ ]:
model = AutoModel.from_pretrained("google/mobilebert-uncased")
model.classifier = nn.Linear(model.config.hidden_size, 1)
model = get_peft_model(model, LoraConfig(r=64, lora_alpha=32, target_modules=["query", "value"], lora_dropout=0.05, bias="none"))
model.to(device)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

In [ ]:
def train(model, train_loader):
    model.train()
    total_loss, correct = 0, 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch).logits.squeeze(1)  # Ensure correct shape
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += ((outputs > 0).float() == y_batch).sum().item()
    return total_loss / len(train_loader), correct / len(X_train)

In [ ]:
def validate(model, val_loader):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            outputs = model(X_batch).logits.squeeze(1)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item()
            correct += ((outputs > 0).float() == y_batch).sum().item()
    return total_loss / len(val_loader), correct / len(X_val)

In [ ]:
epochs = 50
for epoch in tqdm.trange(epochs, colour="red", desc="Epoch(s)"):
    train_loss, train_acc = train(model, train_loader)
    val_loss, val_acc = validate(model, val_loader)
    
    log_data = {"Epoch": epoch+1, "Train Loss": train_loss, "Train Acc": train_acc, "Val Loss": val_loss, "Val Acc": val_acc}
    wandb.log(log_data)
    
    print(log_data)
    model.save_pretrained(f"checkpoints/trainedMobileBERTLoRA_Epoch_{epoch+1}")

In [ ]:
model.save_pretrained("trainedMobileBERTLoRA_Final")
wandb.finish()